In [11]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

def generate_random_rois(num_regions, width, height, max_size=100):
    rois = []
    for _ in range(num_regions):
        w = random.randint(20, max_size)  # Random width
        h = random.randint(20, max_size)  # Random height
        x = random.randint(0, width - w)  # Ensure ROI is within bounds
        y = random.randint(0, height - h)  # Ensure ROI is within bounds
        rois.append((x, y, w, h))
    return rois


def generate_random_shapes(num_shapes, width, height, max_size=100):
    """
    Generates a list of random shapes with diverse prototypes.

    Args:
        num_shapes (int): Number of random shapes to generate.
        width (int): Width of the image.
        height (int): Height of the image.
        max_size (int): Maximum size for the shapes.

    Returns:
        list: List of shape specifications. Each entry is a dictionary with:
            - "type": Type of shape ("rectangle", "ellipse", "circle", "triangle").
            - "points": Points defining the shape (varies by type).
    """
    shapes = []
    for _ in range(num_shapes):
        shape_type = random.choice(["rectangle", "ellipse", "circle", "triangle"])
        if shape_type == "rectangle":
            # Generate rectangle
            w = random.randint(20, max_size)
            h = random.randint(20, max_size)
            x = random.randint(0, width - w)
            y = random.randint(0, height - h)
            shapes.append({"type": "rectangle", "points": (x, y, w, h)})

        elif shape_type == "ellipse":
            # Generate ellipse
            center_x = random.randint(0, width)
            center_y = random.randint(0, height)
            axes_w = random.randint(10, max_size // 2)
            axes_h = random.randint(10, max_size // 2)
            angle = random.randint(0, 360)
            shapes.append({"type": "ellipse", "points": (center_x, center_y, axes_w, axes_h, angle)})

        elif shape_type == "circle":
            # Generate circle
            center_x = random.randint(0, width)
            center_y = random.randint(0, height)
            radius = random.randint(10, max_size // 2)
            shapes.append({"type": "circle", "points": (center_x, center_y, radius)})

        elif shape_type == "triangle":
            # Generate triangle
            pt1 = (random.randint(0, width), random.randint(0, height))
            pt2 = (random.randint(0, width), random.randint(0, height))
            pt3 = (random.randint(0, width), random.randint(0, height))
            shapes.append({"type": "triangle", "points": [pt1, pt2, pt3]})

    return shapes


# Function to apply Gaussian Blur to a region
def apply_gaussian_blur_region(img, roi):
    x, y, w, h = roi
    region = img[y:y+h, x:x+w]
    blurred_region = cv2.GaussianBlur(region, (15, 15), 0)
    img[y:y+h, x:x+w] = blurred_region
    return img


# Function to apply Overexposure to a region
def apply_overexposure_region(img, roi, alpha=1.5, beta=50):
    x, y, w, h = roi
    region = img[y:y+h, x:x+w]
    overexposed_region = cv2.convertScaleAbs(region, alpha=alpha, beta=beta)
    img[y:y+h, x:x+w] = overexposed_region
    return img


def mask_region(image, roi, mask_color=(0, 0, 0), retain_region=False):
    """
    Masks a region in the image with a specified color or retains only the region.

    Args:
        image (np.ndarray): The input image.
        roi (tuple): Region of interest defined as (x, y, width, height).
        mask_color (tuple): RGB/BGR color for masking the region (default: black).
        retain_region (bool): If True, keeps only the region inside the ROI and masks everything else.

    Returns:
        np.ndarray: Image with the region masked.
    """
    x, y, w, h = roi
    
    # Create a mask with the same dimensions as the image
    mask = np.zeros_like(image, dtype=np.uint8)
    
    # Fill the mask with the specified color in the ROI
    if retain_region:
        mask[y:y+h, x:x+w] = 255  # White mask for the ROI
        result = cv2.bitwise_and(image, mask)  # Keep only the masked region
    else:
        mask[y:y+h, x:x+w] = mask_color  # Apply mask color to ROI
        result = image.copy()
        result[y:y+h, x:x+w] = mask_color  # Fill the ROI with the mask color

    return result

In [ ]:
import cv2
import numpy as np

# Load the image
image = cv2.imread('image.jpg')  # Replace with your image path

# Function to create a motion blur kernel
def motion_blur_kernel(size, angle):
    """
    Creates a motion blur kernel of a given size and angle.

    Args:
        size (int): The size of the kernel (must be odd).
        angle (float): The angle of motion in degrees (0° is horizontal).

    Returns:
        np.ndarray: Motion blur kernel.
    """
    kernel = np.zeros((size, size))
    center = size // 2
    
    # Create a line in the kernel
    for i in range(size):
        x = int(center + (i - center) * np.cos(np.radians(angle)))
        y = int(center + (i - center) * np.sin(np.radians(angle)))
        if 0 <= x < size and 0 <= y < size:
            kernel[y, x] = 1
    
    # Normalize the kernel
    kernel /= kernel.sum()
    return kernel

# Function to apply motion blur
def apply_motion_blur(img, size=15, angle=0):
    """
    Applies motion blur to an image using a kernel.

    Args:
        img (np.ndarray): The input image.
        size (int): Kernel size for the motion blur.
        angle (float): Angle of motion in degrees.

    Returns:
        np.ndarray: Image with motion blur applied.
    """
    kernel = motion_blur_kernel(size, angle)
    blurred = cv2.filter2D(img, -1, kernel)
    return blurred

# # Parameters for motion blur
# kernel_size = 21  # Size of the blur kernel
# motion_angle = 30  # Angle of motion in degrees

# # Apply motion blur
# motion_blurred_image = apply_motion_blur(image, size=kernel_size, angle=motion_angle)

# # Show and save the result
# cv2.imshow('Original Image', image)
# cv2.imshow('Motion Blurred Image', motion_blurred_image)
# cv2.imwrite('motion_blurred_image.jpg', motion_blurred_image)

# cv2.waitKey(0)
# cv2.destroyAllWindows()


In [12]:
rgb_path = "/data_nvme/sph/EventScape_processed/Town01-03_train/Town01/sequence_0/rgb/data/01_000_0000_image.png"
image = cv2.imread(rgb_path)
height, width, _ = image.shape

# Generate random areas to apply effects
num_regions = 5  # Number of random regions
# random_rois = generate_random_rois(num_regions, width, height)
random_rois = generate_random_shapes(num_regions, width, height)

# Create a copy of the original image
mask_image = image.copy()
blur_image = image.copy()
overexpose_image = image.copy()

for roi in random_rois:
    # Randomly choose between blur or overexposure
    blur_image = apply_gaussian_blur_region(blur_image, roi)
    overexpose_image = apply_overexposure_region(overexpose_image, roi, alpha=1.8, beta=75)
    mask_image = mask_region(mask_image, roi, mask_color=(0, 0, 0), retain_region=False)

# Draw rectangles around affected areas for visualization
for roi in random_rois:
    x, y, w, h = roi
    cv2.rectangle(blur_image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green rectangle
    cv2.rectangle(overexpose_image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green rectangle
    cv2.rectangle(mask_image, (x, y), (x+w, y+h), (0, 255, 0), 2)  # Green rectangle


# cv2.imshow('Original Imgae', image)
print(image.shape)
plt.imshow(image)
plt.axis("off")
plt.show()

plt.imshow(blur_image)
plt.axis("off")
plt.show()

plt.imshow(overexpose_image)
plt.axis("off")
plt.show()

plt.imshow(mask_image)
plt.axis("off")
plt.show()

ValueError: not enough values to unpack (expected 4, got 2)